# sLLM

**소형 언어 모델(sLLM)**은 매개변수 수를 수십억에서 수백억 대로 제한하여, 대형 언어 모델(LLM)이 제공하는 핵심 기능을 유지하면서도 경량화·고효율화를 추구한 언어 모델이다.  

- **경량화**: 수십억~수백억 규모의 파라미터로 구성되어, LLM 대비 메모리·연산 요구량이 대폭 감소한다.  
- **비용 효율성**: 학습·추론 비용이 낮고, 전력 소모가 적어 온프레미스나 엣지 환경에서 활용하기 적합하다.  
- **실시간성**: 파라미터 수가 적어 추론 속도가 빠르며, 모바일·노트북·임베디스 환경에서도 운영 가능하다.  
- **도메인 특화**: 특정 분야 데이터로 미세조정(fine-tuning)하면 LLM과 유사한 성능을 달성할 수 있다.  

**1. 정의 및 분류**

소형 언어 모델은 파라미터 수가 수십억~수백억 대이며, 일반적으로 다음 세 가지 범주로 구분된다:  
- LLM(매개변수 ≥1천억): 최고 성능·높은 자원 요구  
- sLLM(수십~수백억): 성능·자원 효율 균형  
- sLM(수십억 이하): 단순 작업·극한 제약 환경 특화  

|구분   |파라미터 수       |주요 특징                         |
|-------|------------------|-----------------------------------|
|LLM    |≥1,000억          |최고 성능·고자원                  |
|sLLM   |수십~수백억       |경량화·비용 효율성·응답 속도 우수 |
|sLM    |수십억 이하       |극단적 경량화·제한적 기능         |

**2. 주요 특징**

1. **경량화**  
   - 파라미터 수 감소로 모델 크기·메모리 점유율 최소화  
   - 예: TinyLlama는 11억 파라미터, 4비트 양자화 시 550 MB RAM 차지.  

2. **비용·자원 효율성**  
   - GPU·클라우드 비용 절감  
   - 온프레미스·모바일 환경에서도 실시간 추론 가능.  

3. **도메인 특화 및 미세조정**  
   - 한국어·금융·의료 등 특정 분야 데이터로 추가 학습해 고성능 발휘  
   - 예: beomi/Yi-Ko-6B(6 B 매개변수) 모델은 한국어·영어 병합 데이터로 학습되어 한국어 작업에서 우수한 성능 보유.  

4. **응답 속도**  
   - 매개변수 수가 적어 LLM 대비 추론 속도 2~10배 향상  

**3. 대표적인 sLLM 사례**

|모델명           |파라미터 수 |특징                                                         |
|----------------|----------|-------------------------------------------------------------|
|TinyLlama       |1.1 B     |Llama2 아키텍처 기반, FlashAttention 적용, 4비트 양자화 가능.   |
|Yi-Ko-6B        |6 B       |한국어·영어 혼합 사전학습, 4 K 컨텍스트 길이, HuggingFace 지원.  |
|Mistral 7B      |7.3 B     |영어·코딩 작업 특화, MMLU 60.1% 기록, Apache 2.0 라이선스 공개.    |
|Phi-2           |2.7 B     |MS 온디바이스 AI용, 추론 최적화, 완전 오픈소스.             |
|Motif 2.6B      |2.6 B     |국산 모델, Mistral 7B 대비 134% 우수 성능(지디넷).            |

**4. 모델 구성 및 학습 기법**

- **양자화(Quantization)**: 4~8비트 정밀도 사용으로 메모리 절감  
- **LoRA(Low-Rank Adaptation)**: 저차원 업데이트로 미세조정 시 메모리 효율 극대화  
- **지식 증류(Knowledge Distillation)**: LLM을 교사로 활용해 sLLM에 핵심 지식 전달.  
- **체인 오브 파이팅(CoT) 기법**: 단계별 추론 경로 최적화로 sLLM의 복잡 추론 능력 보완  

**5. 과제 및 한계**

- **추론 능력**: 창의적·복합 문제 해결에서 LLM 대비 성능 격차 존재  
- **언어별 편향**: 한국어·소저자원 언어에서 데이터 부족 시 성능 저하 가능  
- **모델 안전성**: 판별 어려운 환각 위험 관리 필요

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")
os.environ['TAVILY_API_KEY'] = os.getenv("tavily_key")
KAKAO_API_KEY = os.getenv("KAKAO_API_KEY")

os.environ['HF_TOKEN'] = os.getenv("HF_TOKEN")
OPEN_API_KEY = os.environ['OPENAI_API_KEY']
HF_TOKEN = os.environ['HF_TOKEN']


## meta-llama/Llama-3.2-3B-Instruct
https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM  # 허깅페이스에서 토크나이저/언어모델 로드 클래스
import torch  # 텐서 연산 및 GPU 사용을 위한 PyTorch

model_id = 'meta-llama/Llama-3.2-3B-Instruct'  # 불러올 사전학습 Llama 3.2 3B Instruct 모델 이름(허깅페이스 레포 ID)

tokenizer = AutoTokenizer.from_pretrained(model_id)  # 해당 모델에 맞는 토크나이저 다운로드 및 로드(텍스트 → 토큰 변환기)

model = AutoModelForCausalLM.from_pretrained(  # 텍스트 생성용(Causal LM) 모델 가중치 로드
    model_id,                                  # 불러올 모델 레포 ID
    dtype=torch.bfloat16,                      # 가중치를 bfloat16으로 로드 → VRAM 사용량 감소
    device_map='auto'                          # 사용 가능한 GPU/CPU에 자동으로 레이어 분산 배치
)


c:\Users\Playdata\llm\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--meta-llama--Llama-3.2-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 254/254 [00:00<00:00, 567.35it/s, Materializing param=model.norm.wei

In [15]:
model.device

device(type='cpu')

In [17]:
prompt = 'Explain the theory of relativity in simple terms.'            # 모델에게 보낼 입력 문장(프롬프트)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)        # 문장을 토큰 텐서로 변환 후 모델이 올라간 장치(GPU/CPU)로 이동
print(inputs)                                                            # 토큰화된 입력(id, attention_mask 등) 확인

outputs = model.generate(**inputs, max_length=1024)                      # 현재 입력 토큰을 기반으로 최대 1024 토큰 길이까지 이어서 텍스트 생성
print(outputs)                                                           # 생성된 토큰 시퀀스(id 배열) 출력

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'input_ids': tensor([[128000,    849,  21435,    279,  10334,    315,   1375,  44515,    304,
           4382,   3878,     13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


NameError: name 'output' is not defined

In [18]:
outputs

tensor([[128000,    849,  21435,    279,  10334,    315,   1375,  44515,    304,
           4382,   3878,     13,  10771,    311,  55152,     11,   1148,    374,
            279,   5133,   1990,   3634,    323,    892,   5380,     36,  37494,
            596,  10334,    315,   1375,  44515,  29991,    279,   8776,   8830,
            315,   3634,    323,    892,     13,    763,   4382,   3878,     11,
            279,  10334,   5415,    430,   3634,    323,    892,    527,    539,
           8821,  15086,     11,    719,    527,  99892,    439,    264,   3254,
           5502,   2663, 100108,   4199,     13,   5810,    596,    264,  44899,
          16540,    512,    334,   3923,    374, 100108,   4199,     30,   1035,
           6540,    582,   4199,    374,    264,   3116,  33520,  13354,    430,
          33511,   3634,    323,    892,     13,  38891,    264,   6710,    315,
           5684,    449,   1403,  15696,    320,   4222,    323,   2430,      8,
            323,    832,  13

In [20]:
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response


"Explain the theory of relativity in simple terms. According to Einstein, what is the relationship between space and time?\nEinstein's theory of relativity challenged the traditional understanding of space and time. In simple terms, the theory states that space and time are not separate entities, but are intertwined as a single entity called spacetime. Here's a simplified explanation:\n**What is spacetime?**\nSpacetime is a four-dimensional fabric that combines space and time. Imagine a piece of paper with two dimensions (length and width) and one dimension (time). This paper represents the fabric of spacetime, where every event, from the motion of objects to the passage of time, is embedded.\n\n**Key aspects of the theory:**\n\n1. **The speed of light is constant**: No matter where you are or how fast you're moving, the speed of light remains the same. This is a fundamental principle that underlies the theory of relativity.\n2. **Time dilation**: Time appears to pass slower for an obs

In [21]:
def generate_by_sllm(prompt:str):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs,max_length=1024)
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

generate_by_sllm('안녕! 오늘 날씨는 어때?')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'안녕! 오늘 날씨는 어때? (Annyeong! Oyl nal-si-neo neot? - Hello! How\'s the weather today?)\n\n안녕! 날씨는 오늘 너무 춥고 추운 날씨입니다. (Annyeong! Nal-si-neo neot, oyl nam-ju chup-go, chug-eun nal-si-i ni-mas-siyo - Hello! The weather is too cold and windy today.)\n\n아, 그만큼의 온도는? (A, geu-man-kwon-ui on-do neo-yo? - How cold is it?)\n\n그만큼의 온도는 10도 below zero. (Geu-man-kwon-ui on-do neo-yo, 10-do beo-lo zero - It\'s 10 degrees below zero.)\n\n그럼, 오늘은 어때? (Geu-reom, oyl-eun e-ot-deo? - So, how are you?)\n\n어때? 오늘은 날씨가 너무 춥고 추운 날씨로, 나는 나의 집에서만 집중할 수 있습니다. (Eo-tteo? oyl-eun nal-si-neo neot, nam-ju chup-go, chug-eun nal-si-i ro, na-eun na-i ji-seup-eo man ji-seup-eo hal su eum-ni-da - I\'m fine. The weather is too cold and windy today, so I can only focus on my home.)\n\n어떤 날씨가 좋을까요? (Ett-eot nal-si-neo ga jo-ttol-kyeo? - What kind of weather would be good?)\n\n어떤 날씨가 좋을까요? 어른의 날씨가 좋을 때는, 하늘이 구름이 많고, 일기표에 표시된 기온은 15도에서 25도 사이입니다. (Ett-eot nal-si-neo ga jo-ttol-kyeo? eo-reun-i nal-si-neo ga jo-ttol-kyeo, hap-eul-

In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

c:\Users\Playdata\llm\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--Bllossom--llama-3.2-Korean-Bllossom-3B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 254/254 [00:0

In [25]:
instruction = "철수가 20개의 연필을 가지고 있었는데 영희가 절반을 가져가고 민수가 남은 5개를 가져갔으면 철수에게 남은 연필의 갯수는 몇개인가요?"

messages = [
    {"role": "user", "content": f"{instruction}"}
    ]

enc = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True, # 어시스턴트 응답 생성을 위한 프롬프트를 뒤에 추가
    return_tensors="pt",        # Pytorch 텐서로 변환
    return_dict = True          # 딕셔너리 형태로 반환(input_ids, attention_mask등)
).to(model.device)

enc = {k: v.to(model.device) for k, v, in enc.items()}  # 모든 입력 텐서를 모델 디바이스로 이동

print(enc)  # 인코딩된 딕셔너리 구조 확인
print(tokenizer.decode(enc['input_ids'][0], skip_special_tokens=False)) # 만들어진 프롬프트 디코딩해서 확인

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    605,  13806,    220,   2366,     21,    271,   2675,    527,
            264,  11190,  15592,  18328,     13,   5321,   4320,    279,   1217,
            596,   4860,  47626,     13, 113783,  34804, 101003,  67119,  24486,
          15592, 101139,  30426,  25941,  95252,  29726, 119519,     13,  41820,
         110257, 109760,  19954, 112107, 108280, 104834, 102893, 111964,  34983,
          92769,     13, 128009, 128006,    882, 128007,    271, 107837, 123503,
            220,    508, 123590,  78453, 110174,  18359, 120693, 107417, 103170,
         101603, 105204,  20565, 110217, 101738,  18359,  89946,  20565,  35495,
         107138, 123503, 102484,  34804,    220,     20, 117594,  89946,  14705,
            242,  91040, 112521,  24140, 102244, 102484,  34804,  78453, 110174,
          2102

In [ ]:
# 생성 종료로 사용할 토큰 ID 목록
terminators = [
    tokenizer.convert_tokens_to_ids("<|end_of_text|>"), # 문서 종료 토큰 ID
    tokenizer.convert_tokens_to_ids("<|eot_id|>")       # 대화 턴 종료 토큰 ID
]

outputs = model.generate(
    **enc,                                  # 인코딩 dict를 키워드 인자로 풀어서 전달(input_ids, attention_mask등)
    max_new_tokens = 1024,                  # 새로 생성할 최대 토큰 수        
    eos_token_id = terminators,             # 종료 토큰 설정(ID)
    do_sample = True,                       # 샘플링 기반 생성 여부
    temperature = 0.6,                      # 다양성 조절
    top_p =0.9                              # 누적 확률 기반 샘플링 범위
)

prompt_len = enc['input_ids'].shape[-1]     # 입력 프롬프트 길이(토큰 수) 계산

# 프롬프트 이후 생성된 부분만 디코딩해서 출력
print(tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=False)) 

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM    # 토크나이저/생성모델 자동 로더
import torch

model_id = 'LGAI-EXAONE/EXAONE-4.0-1.2B'

tokenizer = AutoTokenizer.from_pretrained(model_id) # 모델이 맞는 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained(       # Cassal LM(생성형) 모델 로드
    model_id,
    torch_dtype=torch.bfloat16,   # bfloat16으로 메모리 절약
    device_map ='auto'    
)

c:\Users\Playdata\llm\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--LGAI-EXAONE--EXAONE-4.0-1.2B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 332/332 [00:01<00:00, 220.38it/s, Materializing param=model.norm.weight] 